In [6]:
# ============================================================
# 06A.1 — FEATURE-DEFINITION AND WINDOW ACCOUNTING AUDIT
# ============================================================
import pandas as pd
import numpy as np  
WINDOW_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "acoustic_features_window.csv"
)

window_features = pd.read_csv(WINDOW_FEATURE_PATH)

print("=" * 70)
print("WINDOW-LEVEL FEATURE AUDIT")
print("=" * 70)

print("Shape:", window_features.shape)
print("Unique recordings:", window_features["recording_id"].nunique())

# ------------------------------------------------------------
# 1. Identify acoustic feature columns
# ------------------------------------------------------------

WINDOW_METADATA_COLS = [
    "state",
    "recording_id",
    "species_id",
    "common_name",
    "scientific_name",
    "window_index",
    "window_start_seconds",
    "window_end_seconds",
    "window_duration_used",
    "flag_short_window",
    "original_sample_rate",
    "original_channels",
    "flag_short_duration",
    "flag_low_amplitude",
    "flag_clipping",
    "flag_low_quality",
    "flag_any",
    "flag_bandlimit_incompatible",
]

WINDOW_ACOUSTIC_FEATURES = [
    c for c in window_features.columns
    if c not in WINDOW_METADATA_COLS
]

print("\nAcoustic feature columns:", len(WINDOW_ACOUSTIC_FEATURES))

# ------------------------------------------------------------
# 2. Valid-window audit
# ------------------------------------------------------------

window_features["_valid_all_features"] = (
    np.isfinite(
        window_features[WINDOW_ACOUSTIC_FEATURES]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy()
    ).all(axis=1)
)

window_counts = (
    window_features
    .groupby("recording_id")
    .agg(
        n_windows=("window_index", "size"),
        n_valid_windows=("_valid_all_features", "sum"),
        duration_used_seconds=("window_duration_used", "sum"),
    )
)

window_counts["all_windows_valid"] = (
    window_counts["n_windows"]
    == window_counts["n_valid_windows"]
)

print("\nWindow validity:")
display(
    window_counts[
        ["n_windows", "n_valid_windows", "all_windows_valid"]
    ]
    .value_counts()
    .rename("recordings")
    .reset_index()
)

print(
    "\nRecordings with any invalid feature window:",
    (~window_counts["all_windows_valid"]).sum()
)

# ------------------------------------------------------------
# 3. Recording duration coverage
# ------------------------------------------------------------

if "master_metadata" not in globals():
    master_metadata = pd.read_csv(
        PROJECT_ROOT
        / "data"
        / "processed"
        / "master_metadata.csv"
    )

duration_audit = (
    master_metadata[
        ["recording_id", "duration_seconds", "species_id", "state"]
    ]
    .copy()
)

duration_audit["recording_id"] = (
    duration_audit["recording_id"].astype(str)
)

window_counts.index = window_counts.index.astype(str)

duration_audit = duration_audit.merge(
    window_counts.reset_index(),
    on="recording_id",
    how="left"
)

duration_audit["retained_fraction"] = (
    duration_audit["duration_used_seconds"]
    / duration_audit["duration_seconds"]
)

duration_audit["discarded_seconds"] = (
    duration_audit["duration_seconds"]
    - duration_audit["duration_used_seconds"]
)

print("\n" + "=" * 70)
print("RECORDING DURATION COVERAGE")
print("=" * 70)

print(
    duration_audit[
        [
            "duration_seconds",
            "duration_used_seconds",
            "discarded_seconds",
            "retained_fraction",
        ]
    ]
    .describe()
    .round(3)
)

print("\nLowest retained-duration recordings:")
display(
    duration_audit[
        [
            "recording_id",
            "state",
            "species_id",
            "duration_seconds",
            "duration_used_seconds",
            "discarded_seconds",
            "retained_fraction",
            "n_windows",
        ]
    ]
    .sort_values("retained_fraction")
    .head(15)
    .round(3)
)

# ------------------------------------------------------------
# 4. NDSI algebra audit
# ------------------------------------------------------------

ndsi_check = window_features[
    ["ndsi", "ndsi_ratio_ba", "anthropophony", "biophony"]
].copy()

ndsi_check["ndsi_reconstructed"] = (
    (ndsi_check["biophony"] - ndsi_check["anthropophony"])
    / (
        ndsi_check["biophony"]
        + ndsi_check["anthropophony"]
    )
)

ndsi_check["ratio_reconstructed"] = (
    ndsi_check["biophony"]
    / ndsi_check["anthropophony"]
)

ndsi_check["ndsi_abs_error"] = (
    ndsi_check["ndsi"]
    - ndsi_check["ndsi_reconstructed"]
).abs()

ndsi_check["ratio_relative_error"] = (
    (
        ndsi_check["ndsi_ratio_ba"]
        - ndsi_check["ratio_reconstructed"]
    ).abs()
    / ndsi_check["ratio_reconstructed"].abs().clip(lower=1e-12)
)

print("\n" + "=" * 70)
print("NDSI ALGEBRA CONSISTENCY")
print("=" * 70)

print(
    "Maximum NDSI reconstruction error:",
    ndsi_check["ndsi_abs_error"].max()
)

print(
    "Median NDSI reconstruction error:",
    ndsi_check["ndsi_abs_error"].median()
)

print(
    "Maximum ratio relative error:",
    ndsi_check["ratio_relative_error"].max()
)

print(
    "Median ratio relative error:",
    ndsi_check["ratio_relative_error"].median()
)

# ------------------------------------------------------------
# 5. Low-frequency biological contamination screening
# ------------------------------------------------------------

# Fraction of total NDSI-band energy in 0–1 kHz.
# This is NOT assumed to be anthropogenic.
window_features["low_band_energy_fraction"] = (
    window_features["anthropophony"]
    / (
        window_features["anthropophony"]
        + window_features["biophony"]
    )
)

low_band_by_species = (
    window_features
    .groupby("species_id")
    .agg(
        n_windows=("recording_id", "size"),
        median_low_band_fraction=(
            "low_band_energy_fraction",
            "median"
        ),
        median_anthropophony=(
            "anthropophony",
            "median"
        ),
        median_biophony=(
            "biophony",
            "median"
        ),
    )
    .reset_index()
)

# Join number of recordings per species
species_recording_counts = (
    master_metadata
    .groupby("species_id")
    .size()
    .rename("n_recordings")
    .reset_index()
)

low_band_by_species = low_band_by_species.merge(
    species_recording_counts,
    on="species_id",
    how="left"
)

print("\n" + "=" * 70)
print("LOW-FREQUENCY (0–1 kHz) SCREENING")
print("=" * 70)

print(
    "Species with >=2 recordings:",
    (low_band_by_species["n_recordings"] >= 2).sum()
)

display(
    low_band_by_species[
        low_band_by_species["n_recordings"] >= 2
    ]
    .sort_values(
        "median_low_band_fraction",
        ascending=False
    )
    .head(20)
    .round(4)
)

# ------------------------------------------------------------
# 6. Band-limit compatibility audit
# ------------------------------------------------------------

bandlimit_audit = (
    master_metadata[
        [
            "recording_id",
            "state",
            "species_id",
            "duration_seconds",
        ]
    ]
    .copy()
)

bandlimit_audit["original_sample_rate"] = (
    window_features
    .groupby("recording_id")["original_sample_rate"]
    .first()
    .reindex(
        bandlimit_audit["recording_id"].astype(str)
    )
    .to_numpy()
)

bandlimit_audit["flag_should_be_true"] = (
    bandlimit_audit["original_sample_rate"] < 20000
)

print("\n" + "=" * 70)
print("1–10 kHz BAND-LIMIT COMPATIBILITY")
print("=" * 70)

display(
    bandlimit_audit[
        bandlimit_audit["flag_should_be_true"]
    ]
)

WINDOW-LEVEL FEATURE AUDIT
Shape: (1179, 69)
Unique recordings: 275

Acoustic feature columns: 51

Window validity:


,n_windows,n_valid_windows,all_windows_valid,recordings
0,1,1,True,117
1,2,2,True,39
2,3,3,True,25
3,4,4,True,22
4,5,5,True,15
5,6,6,True,12
6,7,7,True,7
7,8,8,True,5
8,11,11,True,5
9,10,10,True,4



Recordings with any invalid feature window: 0

RECORDING DURATION COVERAGE
       duration_seconds  duration_used_seconds  discarded_seconds  \
count           275.000                275.000            275.000   
mean             46.038                 42.078              3.961   
std              64.861                 64.323              3.231   
min               0.696                  0.696             -0.000   
25%              12.864                 10.000              0.632   
50%              23.975                 20.000              3.848   
75%              54.105                 50.000              6.781   
max             646.992                640.000              9.800   

       retained_fraction  
count            275.000  
mean               0.867  
std                0.141  
min                0.509  
25%                0.805  
50%                0.906  
75%                0.987  
max                1.000  

Lowest retained-duration recordings:


,recording_id,state,species_id,duration_seconds,duration_used_seconds,discarded_seconds,retained_fraction,n_windows
105,728080,AZ,Dendrocygna autumnalis,19.653,10.0,9.653,0.509,1
69,17795,CA,Mareca americana,19.481,10.0,9.481,0.513,1
197,218546,TX,Spatula discors,19.477,10.0,9.477,0.513,1
113,511453,AZ,Branta canadensis,19.369,10.0,9.369,0.516,1
60,552939,CA,Aeronautes saxatalis,19.224,10.0,9.224,0.520,1
2,452094,CA,Fulica americana,18.960,10.0,8.960,0.527,1
155,611978,AZ,Spatula cyanoptera,18.864,10.0,8.864,0.530,1
152,132890,AZ,Selasphorus platycercus,18.730,10.0,8.730,0.534,1
147,491896,AZ,Patagioenas fasciata,18.672,10.0,8.672,0.536,1
266,233658,TX,Lanius ludovicianus,18.484,10.0,8.484,0.541,1



NDSI ALGEBRA CONSISTENCY
Maximum NDSI reconstruction error: 2.7755575615628914e-16
Median NDSI reconstruction error: 8.673617379884035e-18
Maximum ratio relative error: 2.297841972369956e-13
Median ratio relative error: 0.0

LOW-FREQUENCY (0–1 kHz) SCREENING
Species with >=2 recordings: 56


,species_id,n_windows,median_low_band_fraction,median_anthropophony,median_biophony,n_recordings
93,Laterallus jamaicensis,8,0.9962,3.146989e+05,1825.6512,3
53,Coccyzus americanus,34,0.9928,1.326247e+04,56.5404,2
118,Patagioenas fasciata,12,0.9767,9.059040e+04,330.8291,4
56,Columbina inca,5,0.9644,1.820711e+05,16044.6366,3
51,Chordeiles acutipennis,33,0.9626,2.794743e+04,632.2439,4
145,Streptopelia roseogrisea,2,0.9549,2.601269e+04,866.5994,2
19,Antrostomus vociferus,38,0.9542,4.006213e+06,218326.5659,3
144,Streptopelia decaocto,16,0.9489,5.294436e+04,430.2447,2
105,Meleagris gallopavo,26,0.8774,8.533612e+04,4347.0578,2
81,Fulica americana,8,0.8451,2.997325e+04,5934.7851,3



1–10 kHz BAND-LIMIT COMPATIBILITY


,recording_id,state,species_id,duration_seconds,original_sample_rate,flag_should_be_true


In [2]:
# ============================================================
# 06A.0 — PROJECT PATH SETUP
# ============================================================

from pathlib import Path

# Notebook 06 is located at:
# research/notebooks/06_PCA_Clustering_Analysis.ipynb

PROJECT_ROOT = Path.cwd().parent.parent

print("Current working directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nProject root exists:", PROJECT_ROOT.exists())
print("Processed directory exists:",
      (PROJECT_ROOT / "data" / "processed").exists())

Current working directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\research\notebooks

Project root:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity

Project root exists: True
Processed directory exists: True


In [3]:
WINDOW_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "acoustic_features_window.csv"
)

print(WINDOW_FEATURE_PATH)
print("Exists:", WINDOW_FEATURE_PATH.exists())

c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\processed\acoustic_features_window.csv
Exists: True


In [7]:
# ============================================================
# 06A.2 — BAND-LIMIT COMPATIBILITY SUMMARY
# ============================================================

bandlimit_summary = (
    bandlimit_audit
    .groupby("flag_should_be_true")
    .agg(
        n_recordings=("recording_id", "size"),
        median_sample_rate=("original_sample_rate", "median"),
        min_sample_rate=("original_sample_rate", "min"),
        max_sample_rate=("original_sample_rate", "max"),
    )
    .reset_index()
)

print("=" * 70)
print("1–10 kHz BAND-LIMIT COMPATIBILITY SUMMARY")
print("=" * 70)

display(bandlimit_summary)

print(
    "\nRecordings incompatible with complete 1–10 kHz band:",
    bandlimit_audit["flag_should_be_true"].sum()
)

print(
    "Compatible recordings:",
    (~bandlimit_audit["flag_should_be_true"]).sum()
)

print(
    "Percentage incompatible:",
    round(
        100 * bandlimit_audit["flag_should_be_true"].mean(),
        2
    ),
    "%"
)

1–10 kHz BAND-LIMIT COMPATIBILITY SUMMARY


,flag_should_be_true,n_recordings,median_sample_rate,min_sample_rate,max_sample_rate
0,False,275,NaN,NaN,NaN



Recordings incompatible with complete 1–10 kHz band: 0
Compatible recordings: 275
Percentage incompatible: 0.0 %
